# 02.04 — Final reporting

Câu hỏi: Core đã freeze cho kết quả báo cáo thế nào trên hai final set? Mở notebook này chỉ sau `python tools/validate_phase02_final_reporting.py` trả về PASS. Nó không chạy evaluation và không tạo lựa chọn mới.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'configs/phase02_protocol.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Không tìm thấy repository root.')
BASE = ROOT / 'evidence/phase02/final'
decision = json.loads((ROOT / 'evidence/phase02/core_mt_decision.json').read_text(encoding='utf-8'))
assert decision['status'] == 'frozen_before_final_evaluation'
rows = []
for metric_path in BASE.glob('*/*/metrics.json'):
    run_path = metric_path.with_name('run.json')
    if run_path.is_file():
        metric, run = json.loads(metric_path.read_text(encoding='utf-8')), json.loads(run_path.read_text(encoding='utf-8'))
        rows.append({'direction': run['adapter']['direction'], 'dataset': metric['dataset_role'], 'checkpoint': run.get('evaluated_checkpoint'), 'rows': metric['rows'], 'chrF++': metric['chrF++'], 'SacreBLEU': metric['sacreBLEU'], 'runtime': run['adapter'].get('transformers_version'), 'evidence': str(metric_path.relative_to(ROOT))})
final = pd.DataFrame(rows).sort_values(['direction','dataset']) if rows else pd.DataFrame()
final

,direction,dataset,checkpoint,rows,chrF++,SacreBLEU,runtime,evidence
0,en_to_vi,general_test,"{'id': 'final-epoch-1', 'path': 'models/phase0...",1012,60.094073,39.284927,4.57.6,evidence\phase02\final\en_to_vi\general_test\m...
1,en_to_vi,it_test,"{'id': 'final-epoch-1', 'path': 'models/phase0...",15244,70.329635,56.819231,4.57.6,evidence\phase02\final\en_to_vi\it_test\metric...
2,vi_to_en,general_test,"{'id': 'final-epoch-1', 'path': 'models/phase0...",1012,57.734424,28.350020,4.57.6,evidence\phase02\final\vi_to_en\general_test\m...
3,vi_to_en,it_test,"{'id': 'final-epoch-1', 'path': 'models/phase0...",15244,65.210469,41.200832,4.57.6,evidence\phase02\final\vi_to_en\it_test\metric...


In [2]:
expected = {(d, s) for d in ('en_to_vi','vi_to_en') for s in ('it_test','general_test')}
actual = set(zip(final.direction, final.dataset)) if not final.empty else set()
print('Missing final reports:', sorted(expected - actual))
assert not (expected - actual), 'Final reporting is incomplete.'

Missing final reports: []


Báo cáo kết quả tách IT Test và General Test, checkpoint đã khóa và điều kiện chạy thực tế. Không dùng bảng này để đổi Core, checkpoint hoặc hyperparameter.